In [1]:
import tensorflow as tf
import tensorflow.keras.layers as layers
import tensorflow.keras.models as models
import tensorflow_model_optimization as tfmot
import numpy as np
import pathlib

# --- 0. Load Data (From your original script) ---
print("Loading data...")
data_dir = pathlib.Path("/home/theorist/Desktop/Programming/ML/computing_tools/capstone-1/")
data_dir = data_dir / "train" / "images"

batch_size = 32
img_height = 224
img_width = 224

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  color_mode='grayscale'  
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  color_mode='grayscale'  
)

AUTOTUNE = tf.data.AUTOTUNE

# Add .repeat() to your training dataset
train_ds = train_ds.cache().shuffle(1000).repeat().prefetch(buffer_size=AUTOTUNE)

# Validation dataset should NOT repeat
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Data loaded.")

# --- 1. Build a Temporary *Nested* Model ---
# This model's *only* purpose is to act as a
# container to load your .weights.h5 file.
print("Building temporary nested model...")
inputs_temp = layers.Input(shape=(224, 224, 1), name='input_layer_10')
x_temp = layers.Conv2D(3, kernel_size=1, padding='same', name='conv2d_5')(inputs_temp)
x_temp = layers.Rescaling(1./127.5, offset=-1, name='rescaling_5')(x_temp)

# This is the "nested" way to call a model
base_model_temp = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)
x_temp = base_model_temp(x_temp) # <-- Calling the model as a layer

x_temp = layers.GlobalAveragePooling2D(name='global_average_pooling2d_5')(x_temp)
x_temp = layers.Dropout(0.2, name='dropout_5')(x_temp)
outputs_temp = layers.Dense(6, name='dense_4')(x_temp)

model_nested_temp = models.Model(inputs_temp, outputs_temp, name='model_nested_temp')

# --- 2. Load Weights into the Temporary Model ---
print("Loading weights from .weights.h5 file...")
# (Make sure this file path is correct!)
model_nested_temp.load_weights("steel_defect_clf_mobile_netv2_fine_tuning_scaling_patch_compat.weights.h5")
print("Weights loaded into temp model.")

# --- 3. Build the *Truly Flat* Model for TFMOT ---
print("Building *truly flat* model...")
inputs_flat = layers.Input(shape=(224, 224, 1), name='input_layer_10')
x_flat = layers.Conv2D(3, kernel_size=1, padding='same', name='conv2d_5')(inputs_flat)
x_flat = layers.Rescaling(1./127.5, offset=-1, name='rescaling_5')(x_flat)

# This is the "flat" way to build a model
# We pass our tensor `x_flat` directly to the `input_tensor` arg
base_model_flat = tf.keras.applications.MobileNetV2(
    input_tensor=x_flat, # <-- This is the key
    include_top=False,
    weights=None
)

# Continue from the *output* of the base model
x_flat_head = layers.GlobalAveragePooling2D(name='global_average_pooling2d_5')(base_model_flat.output)
x_flat_head = layers.Dropout(0.2, name='dropout_5')(x_flat_head)
outputs_flat = layers.Dense(6, name='dense_4')(x_flat_head)

model_truly_flat = models.Model(inputs=inputs_flat, outputs=outputs_flat, name='model_truly_flat')
print("Truly flat model built.")

# --- 4. Manually Transfer Weights to the Flat Model ---
print("Transferring weights from temp model to flat model...")
# Get the nested part of the temp model
nested_mobilenet_layer = model_nested_temp.get_layer('mobilenetv2_1.00_224')

# These are the "outer" layers we will handle separately
outer_layer_names = {
    'input_layer_10', 
    'conv2d_5', 
    'rescaling_5', 
    'global_average_pooling2d_5', 
    'dropout_5', 
    'dense_4'
}

# 1. Copy weights for all INNER MobileNetV2 layers
for layer_flat in model_truly_flat.layers:
    # Skip outer layers
    if layer_flat.name in outer_layer_names:
        continue
        
    # Skip layers with no weights (ReLU, Add, etc.)
    if not layer_flat.get_weights():
        continue
        
    # This layer is an inner layer with weights (Conv, BN, etc.)
    # Copy weights from the corresponding layer in the nested model.
    try:
        nested_layer = nested_mobilenet_layer.get_layer(layer_flat.name)
        layer_flat.set_weights(nested_layer.get_weights())
    except Exception as e:
        print(f"WARNING: Could not copy weights for layer {layer_flat.name}: {e}")

# 2. Copy weights for the "outer" layers
model_truly_flat.get_layer('conv2d_5').set_weights(
    model_nested_temp.get_layer('conv2d_5').get_weights()
)
model_truly_flat.get_layer('dense_4').set_weights(
    model_nested_temp.get_layer('dense_4').get_weights()
)
print("--- Weight transfer complete. ---")


# --- 5. Apply Pruning ---
print("Applying pruning wrappers...")
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

total_train_images = int(1799 * 0.8)
steps_per_epoch = total_train_images // batch_size
epochs = 10
total_steps = steps_per_epoch * epochs

pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.00, final_sparsity=0.50, begin_step=0, end_step=total_steps
    )
}

def apply_pruning_to_layer(layer):
    if isinstance(layer, (tf.keras.layers.Conv2D, 
                           tf.keras.layers.Dense, 
                           tf.keras.layers.DepthwiseConv2D)):
        return prune_low_magnitude(layer, **pruning_params)
    return layer

model_for_pruning = tf.keras.models.clone_model(
    model_truly_flat, # <-- Use the flat model
    clone_function=apply_pruning_to_layer,
)

print("--- Manually forcing weights into pruned model (Layer by Layer)... ---")

for layer_flat, layer_pruned in zip(model_truly_flat.layers, model_for_pruning.layers):
    # Skip layers with no weights (Input, ReLU, Add, etc.)
    if not layer_flat.get_weights():
        continue
        
    if hasattr(layer_pruned, 'layer'):
        # This is a PruneLowMagnitude wrapper.
        # We set the weights on the *inner* layer it's wrapping.
        layer_pruned.layer.set_weights(layer_flat.get_weights())
    else:
        # This is a layer that wasn't pruned (e.g., BatchNormalization).
        # We can set weights directly.
        layer_pruned.set_weights(layer_flat.get_weights())
        
callbacks = [ tfmot.sparsity.keras.UpdatePruningStep() ]
print("--- 'model_for_pruning' is ready. (You must .fit() this model) ---")

2025-11-09 01:14:36.199760: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-09 01:14:36.201324: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-09 01:14:36.224899: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-09 01:14:36.224922: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-09 01:14:36.225438: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Loading data...
Found 1799 files belonging to 6 classes.
Using 1440 files for training.
Found 1799 files belonging to 6 classes.
Using 359 files for validation.
Data loaded.
Building temporary nested model...
Loading weights from .weights.h5 file...
Weights loaded into temp model.
Building *truly flat* model...
Truly flat model built.
Transferring weights from temp model to flat model...
--- Weight transfer complete. ---
Applying pruning wrappers...
--- Manually forcing weights into pruned model (Layer by Layer)... ---
--- 'model_for_pruning' is ready. (You must .fit() this model) ---


In [2]:
print("--- DEBUG: Evaluating flat model accuracy BEFORE pruning... ---")

# Compile the flat model
model_truly_flat.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Calculate validation steps
total_val_images = int(1799 * 0.2)
validation_steps = total_val_images // 1 # Since batch_size is 1

# Evaluate the model
results = model_truly_flat.evaluate(
    val_ds,
    steps=validation_steps
)
print(f"--- DEBUG RESULTS: Loss: {results[0]}, Accuracy: {results[1]} ---")
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# --- END DEBUG STEP ---
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

--- DEBUG: Evaluating flat model accuracy BEFORE pruning... ---
359/359 [==============================] - 3s 5ms/step - loss: 0.0115 - accuracy: 0.9944
--- DEBUG RESULTS: Loss: 0.011490298435091972, Accuracy: 0.9944289922714233 ---


In [3]:
print("--- Starting pruning fine-tuning... ---")
# Use a very low learning rate
model_for_pruning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Low LR
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# You must use the callback
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

# Add this calculation before your model.fit() call
total_train_images = int(1799 * 0.8)
steps_per_epoch = total_train_images // batch_size

# You also need steps for validation
total_val_images = int(1799 * 0.2)
validation_steps = total_val_images // 32 # Since batch_size is 1

# Run fit for a few epochs
model_for_pruning.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=callbacks,
    steps_per_epoch=steps_per_epoch,       # <--- ADD THIS
    validation_steps=validation_steps      # <--- AND THIS
)
print("--- Pruning fine-tuning complete. ---")

--- Starting pruning fine-tuning... ---
Epoch 1/10
 8/44 [====>.........................] - ETA: 56s - loss: 9.9891 - accuracy: 0.1719

KeyboardInterrupt: 

In [2]:
!pip install "tf-keras==2.15.0"
!pip install "tensorflow==2.15.0"
!pip install "tensorflow-model-optimization==0.7.5"

  Using cached tensorflow-2.15.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.4 kB)
Using cached tensorflow-2.15.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (475.2 MB)
  Using cached tensorflow_model_optimization-0.7.5-py2.py3-none-any.whl.metadata (914 bytes)
Using cached tensorflow_model_optimization-0.7.5-py2.py3-none-any.whl (241 kB)
